# 16 A Numerical Study, Built One Step at a Time

| | |
|---|---|
| **Path** | NumPy |
| **Prerequisite** | 15 |

The earlier lessons each added one tool. This lesson uses them on one table of scores. Run the cells from the top. Each cell adds one step and uses the names already defined.

Rows are students. Columns are tests. The table is

$$
\begin{bmatrix}
4 & 6 & 8 \\
10 & 10 & 10 \\
2 & 2 & 5
\end{bmatrix}.
$$

The steps are: build the array, reduce it, mask it, standardize a column by hand, then save and load the result.


## Step 1 — Build the table and read its shape

### Predict

Three students and three tests: shape `(3, 3)`, size 9, dtype an integer kind.

### Run


In [1]:
import numpy as np

# Rows are students. Columns are tests.
scores = np.array([
    [4, 6, 8],
    [10, 10, 10],
    [2, 2, 5],
])
print(scores.shape)
print(scores.size)
print(np.issubdtype(scores.dtype, np.integer))


(3, 3)
9
True


## Step 2 — Means a person can recompute

### Predict

Across each row, the student means are

$$
\dfrac{4+6+8}{3} = 6,
\qquad
\dfrac{10+10+10}{3} = 10,
\qquad
\dfrac{2+2+5}{3} = 3.
$$

Down each column, the test means are

$$
\dfrac{4+10+2}{3} = \dfrac{16}{3},
\qquad
\dfrac{6+10+2}{3} = 6,
\qquad
\dfrac{8+10+5}{3} = \dfrac{23}{3}.
$$

### Run


In [2]:
import numpy as np

# axis 1 removes the test axis, leaving one mean per student.
student_means = scores.mean(axis=1)
test_means = scores.mean(axis=0)
print(student_means)
print(np.allclose(test_means, [16 / 3, 6, 23 / 3]))


[ 6. 10.  3.]
True


## Step 3 — Who reached a mean of at least 6

### Predict

The student means are $6$, $10$, and $3$. The mask `student_means >= 6` keeps the first two rows and drops the third.

$$
\begin{bmatrix}
4 & 6 & 8 \\
10 & 10 & 10
\end{bmatrix}
$$

### Run


In [3]:
import numpy as np

passed = student_means >= 6
print(passed)
# A boolean row mask selects students. It is a copy of those rows.
print(scores[passed])


[ True  True False]
[[ 4  6  8]
 [10 10 10]]


## Step 4 — Standardize the first test by hand

### Predict

The first column is $4, 10, 2$. Its mean is $16/3$. The squared deviations sum as follows:

$$
\begin{aligned}
\left(4 - \dfrac{16}{3}\right)^2
+ \left(10 - \dfrac{16}{3}\right)^2
+ \left(2 - \dfrac{16}{3}\right)^2
&= \left(-\dfrac{4}{3}\right)^2 + \left(\dfrac{14}{3}\right)^2 + \left(-\dfrac{10}{3}\right)^2 \\
&= \dfrac{16 + 196 + 100}{9}
= \dfrac{312}{9}
= \dfrac{104}{3}.
\end{aligned}
$$

That last fraction is $\sum (x_i-\mu)^2 = 104/3$, not yet divided by $n$. The population variance divides by $n=3$:

$$
\sigma^2 = \dfrac{1}{3}\cdot\dfrac{104}{3} = \dfrac{104}{9},
\qquad
z = \dfrac{x-\mu}{\sigma}.
$$

NumPy's default `std` is this $\sigma$. After standardization, the column has mean $0$ and default standard deviation $1$.

### Run


In [4]:
import numpy as np

column = scores[:, 0].astype(float)
mu = column.mean()
sigma = column.std()
by_hand = (column - mu) / sigma
# keepdims lets the column statistics broadcast across nothing here; they are scalars.
print(np.allclose(mu, 16 / 3))
print(np.allclose(sigma ** 2, 104 / 9))
print(np.allclose(by_hand.mean(), 0))
print(np.allclose(by_hand.std(), 1))


True
True
True
True


## Step 5 — Standardize every column, then save the table

### Predict

The same formula on each column, with `keepdims=True`, produces a $3\times 3$ table whose columns each have mean 0 and standard deviation 1. Saving that table and reading it back returns the same numbers.

$$
Z_{ij} = \dfrac{X_{ij} - \mu_j}{\sigma_j}.
$$

### Run


In [5]:
import tempfile
from pathlib import Path
import numpy as np

data = scores.astype(float)
mu = data.mean(axis=0, keepdims=True)
sigma = data.std(axis=0, keepdims=True)
standardized = (data - mu) / sigma
print(standardized.shape)
print(np.allclose(standardized.mean(axis=0), 0))
print(np.allclose(standardized.std(axis=0), 1))
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "scores.npy"
    # save writes the array. load reads that same block back.
    np.save(path, standardized)
    restored = np.load(path)
print(np.allclose(restored, standardized))


(3, 3)
True
True
True


## A pitfall

Standardizing with the sample standard deviation (`ddof=1`) divides by a larger number, so the resulting column does **not** have `std()` equal to 1 under NumPy's default. The default `std` divides by $n$. If you change only the standardization and forget the check, the last line of this cell is false on purpose: it shows the mismatch.

### Run


In [6]:
import numpy as np

column = scores[:, 0].astype(float)
sample = (column - column.mean()) / column.std(ddof=1)
# Default std divides by n. The sample formula divided by n - 1.
print(np.allclose(sample.std(), 1))
print(np.allclose(sample.std(ddof=1), 1))


False
True


## Summary

A table is a two-dimensional array with an axis for each kind of thing you might reduce. Masks select rows. Standardization subtracts a column mean and divides by that column's standard deviation, and `keepdims=True` is what lets those column statistics line up. `np.save` and `np.load` round-trip the array. When you divide by $n-1$, check the deviation with the same `ddof` you used to build it.

This is the last lesson of the NumPy path. You can build an array, reshape it, index it, broadcast it, reduce it, handle missing data, and carry a small linear-algebra calculation from a hand solution back to `@` and `solve`.
